In [ ]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from cmdstanpy import CmdStanModel
import cmdstanpy
import numpy as np
from scipy.stats import invgamma
from scipy.stats import norm
from scipy.special import softmax
from gptools.stan import get_include

import pandas as pd
from glob import glob
from nteprsm import utils 
from cmdstanpy import stanfit
from settings import ROOT_DIR
import plotly.express as px
import utils as notebook_utils
# use customize plotly template
notebook_utils.set_custom_template()

import pickle

In [ ]:
filepath = ROOT_DIR/"QUALITY_NJ2"
df = pd.read_csv('data/raw/quality_nj2.csv')  # Replace 'file.csv' with your file path
df.columns = [col.lower() for col in df.columns]
df = df.assign(
    entry_name_code=pd.Categorical(df["entry_name"]).codes,
    plt_id_code=pd.Categorical(df["plt_id"]).codes,
    rater_code=pd.Categorical(df["rater"]).codes,
    rating_event_code=pd.Categorical(df["rating_event"]).codes,
)
df["entry_cumcount"] = df.groupby("entry_name").cumcount() + 1

In [ ]:
# load model configuration
config_file = ROOT_DIR/"config/nteprsm_njkbg07.yml"
config = utils.load_config(config_file)
config["sampling"]['save_warmup'] = False
# process data
datahandler = utils.DataHandler(filepath='data/raw/quality_nj2.csv')
datahandler.model_data = df
datahandler.load_data()
datahandler.preprocess_data()
datahandler.generate_stan_data(**config["stan_additional_data"])

In [ ]:
config['stan_file'] = 'models/no_consistent_rater_model_dist_matrix_loglik.stan'
config

In [ ]:
# create dist matrix

id_code_to_plt_id = datahandler.model_data.groupby('plt_id_code')['plt_id'].mean().astype(int).to_dict()
plt_id_to_row =  datahandler.model_data.groupby('plt_id')['row'].mean().astype(int).to_dict()
plt_id_to_col = datahandler.model_data.groupby('plt_id')['col'].mean().astype(int).to_dict()
plt_id_to_col

num_plots = datahandler.stan_data['num_plots']
dist_matrix = np.zeros(shape = (num_plots, num_plots))
for i in range(num_plots):
    for j in range(num_plots):
        plt_id_i, plt_id_j = id_code_to_plt_id[i], id_code_to_plt_id[j]
        row_i, row_j = plt_id_to_row[plt_id_i], plt_id_to_row[plt_id_j]
        col_i, col_j = plt_id_to_col[plt_id_i], plt_id_to_col[plt_id_j]
        dist = np.sqrt((row_i - row_j)**2 + (col_i - col_j)**2)
        dist_matrix[i][j] = dist

In [ ]:
datahandler.stan_data["I"] = len(datahandler.model_data['rating_event_code'].unique())
datahandler.stan_data["J"] = datahandler.stan_data['num_entries']
datahandler.stan_data["P"] = datahandler.stan_data['num_plots']
datahandler.stan_data["M"] = datahandler.stan_data['num_categories'] - 1
datahandler.stan_data["ii"] = datahandler.model_data['rating_event_code'] + 1
datahandler.stan_data["jj"] = datahandler.model_data['entry_name_code'] + 1
datahandler.stan_data["pp"] = datahandler.model_data['plt_id_code'] + 1
datahandler.stan_data["y"] =  datahandler.model_data['quality'] - 1

# create dist matrix
id_code_to_plt_id = datahandler.model_data.groupby('plt_id_code')['plt_id'].mean().astype(int).to_dict()
plt_id_to_row =  datahandler.model_data.groupby('plt_id')['row'].mean().astype(int).to_dict()
plt_id_to_col = datahandler.model_data.groupby('plt_id')['col'].mean().astype(int).to_dict()
plt_id_to_col

num_plots = datahandler.stan_data['num_plots']
dist_matrix = np.zeros(shape = (num_plots, num_plots))
for i in range(num_plots):
    for j in range(num_plots):
        plt_id_i, plt_id_j = id_code_to_plt_id[i], id_code_to_plt_id[j]
        row_i, row_j = plt_id_to_row[plt_id_i], plt_id_to_row[plt_id_j]
        col_i, col_j = plt_id_to_col[plt_id_i], plt_id_to_col[plt_id_j]
        dist = np.sqrt((row_i - row_j)**2 + (col_i - col_j)**2)
        dist_matrix[i][j] = dist
datahandler.stan_data["DIST"] = dist_matrix

In [ ]:
nteprsm = CmdStanModel(
    stan_file=config["stan_file"],
    stanc_options={"include-paths": get_include()},
)
fit = nteprsm.sample(data=datahandler.stan_data, **config["sampling"])

In [ ]:
# Save to a file
with open('old_model_fit2', 'wb') as file:
    pickle.dump(fit, file)